# Track 1 Smart Campus - prediction notebook (c00_lgb_baseline)

Inference only: loads the trained artifact `model.pkl` (trained locally, see the training notebook) and predicts `energy_usage` for every row of the input file, in input order. All features are row-local, so shuffled rows / a missing `id` column are fine.

In [ ]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

TEST_PATH = os.environ.get("DATATHON_INPUT_PATH", "test.csv")
OUTPUT_PATH = os.environ.get("DATATHON_OUTPUT_PATH", "predictions.csv")

In [ ]:
# Feature engineering - identical to training (src/features.py)
import numpy as np
import pandas as pd

DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
BUILDINGS = ["ADM_A", "BUS_A", "BUS_B", "ENG_A", "ENG_B", "LEC_A", "LIB_A",
             "RES_A", "RES_B", "SCI_A", "SCI_B", "SPT_A"]
TYPES = ["Administration", "Business", "Engineering", "LectureHall", "Library",
         "Residential", "Science", "Sports"]


def make_features(df):
    df = df.copy()
    out = pd.DataFrame(index=df.index)
    # fixed category lists -> identical integer codes at train and inference; unseen -> NaN
    out["building"] = pd.Categorical(df["building_id"], categories=BUILDINGS)
    out["btype"] = pd.Categorical(df["building_type"], categories=TYPES)
    out["hour"] = df["hour"].astype(float)
    out["dow"] = pd.Categorical(df["day_of_week"], categories=DAYS).codes.astype(float)
    out.loc[out["dow"] < 0, "dow"] = np.nan
    out["weekend"] = (out["dow"] >= 5).astype(float)
    out["month"] = df["month"].astype(float)
    for c in ["temperature", "humidity", "occupancy", "previous_usage"]:
        out[c] = pd.to_numeric(df[c], errors="coerce").astype(float)
    return out

In [ ]:
artifact = joblib.load("model.pkl")
test = pd.read_csv(TEST_PATH)
X_test = make_features(test)[artifact["features"]]
preds = artifact["model"].predict(X_test)
preds = np.asarray(preds, dtype=float).ravel()
assert len(preds) == len(test) and np.isfinite(preds).all()

In [ ]:
pd.DataFrame({"prediction": preds}).to_csv(OUTPUT_PATH, index=False)
print(f"wrote {len(preds)} predictions to {OUTPUT_PATH}")